In [1]:
from itertools import islice
import time

import numpy as np
import pandas as pd

## 1. Wczytanie danych

In [2]:
clicks = pd.concat(
    chunk.loc[chunk["event"] == "click", ["user", "item", "timestamp"]]
    for chunk in pd.read_csv("data/interactions.csv", chunksize=2_000_000)
)

clicks["timestamp"] = pd.to_datetime(clicks["timestamp"], unit="s")

print("liczba kliknięć: ", len(clicks))
print("liczba użytkowników: ", clicks["user"].nunique())
print("liczba ofert: ", clicks["item"].nunique())
print("okres od: ", clicks["timestamp"].min(), " do: ", clicks["timestamp"].max())
clicks.head()

liczba kliknięć:     58817263
liczba użytkowników: 3286174
liczba ofert:        185164
okres:        2020-02-12 00:00:00 - 2020-02-25 23:59:59


,user,item,timestamp
0,27901,56865,2020-02-12
1,124480,115662,2020-02-12
2,159509,5150,2020-02-12
3,188861,109981,2020-02-12
4,207348,88746,2020-02-12


In [17]:
#clicks.to_parquet("data/clicks.parquet", index=False)
clicks = pd.read_parquet("data/clicks.parquet")

## 2. Losowe 10% użytkowników

In [18]:
users = clicks["user"].drop_duplicates().sample(frac=0.1, random_state=42)
sample = clicks[clicks["user"].isin(users)]

print(f"{len(users)} użytkowników, {len(sample)} kliknięć")

328617 użytkowników, 5883756 kliknięć


## 3. Temporal split

In [19]:
t_split = sample["timestamp"].quantile(0.8)

train = sample[sample["timestamp"] < t_split]
test = sample[sample["timestamp"] >= t_split]

print("t_split:", t_split)
print(f"train: {len(train)}, test: {len(test)}")

t_split: 2020-02-22 19:38:10
train: 4707002, test: 1176754


## 4. Katalog: 10% najpopularniejszych ofert

In [20]:
pop = train["item"].value_counts()
catalog = pop.head(int(len(pop) * 0.1)).index

train = train[train["item"].isin(catalog)]
test = test[test["item"].isin(catalog)]

print(f"liczba ofert w katalogu: {len(catalog)}")
print(f"zostało w train: {pop[catalog].sum() / pop.sum():.1%} kliknięć")

print(f"train: {train['user'].nunique()} userów, {train['item'].nunique()} ofert, {len(train)} interakcji")
print(f"test: {test['user'].nunique()} userów, {test['item'].nunique()} ofert, {len(test)} interakcji")

liczba ofert w katalogu: 13317
zostało w train: 44.1% kliknięć
train: 210642 userów, 13317 ofert, 2076316 interakcji
test: 72923 userów, 13257 ofert, 377367 interakcji


In [21]:
# Zostawiamy Ttylko użytkowników, którzy po filtrze katalogu zostali w zbiorze testowym i treningowym. 

train_items = train.groupby("user")["item"].agg(set)
test_items = test.groupby("user")["item"].agg(set)

eval_users = train_items.index.intersection(test_items.index)
test_items = test_items[eval_users]

print(f"liczba użytkowników: {len(eval_users)}")
print(f"średnio ofert w teście: {test_items.map(len).mean():.2f}")

liczba użytkowników: 44317
średnio ofert w teście: 5.07


In [22]:
overlap = [len(test_items[u] & train_items[u]) / len(test_items[u]) for u in eval_users]
print(f"oferty z testu widziane już w train: {np.mean(overlap):.1%}")

oferty z testu widziane już w train: 25.0%


## Metryki

In [23]:
def precision_recall(recs, k=10):
    precision, recall = [], []
    for user in eval_users:
        hits = len(set(recs[user][:k]) & test_items[user])
        precision.append(hits / k)
        recall.append(hits / len(test_items[user]))
    return np.mean(precision), np.mean(recall)

## Baseline

In [24]:
t0 = time.time()
pop_order = train["item"].value_counts().index
fit_time = time.time() - t0


def popularity(users, k=10, exclude_seen=False):
    if not exclude_seen:
        return {u: list(pop_order[:k]) for u in users}
    return {
        u: list(islice((i for i in pop_order if i not in train_items[u]), k))
        for u in users
    }

In [25]:
results = []
for label, exclude_seen in [("nie wykluczamy", False), ("wykluczamy", True)]:
    t0 = time.time()
    recs = popularity(eval_users, exclude_seen=exclude_seen)
    predict_time = time.time() - t0

    precision, recall = precision_recall(recs)
    results.append({
        "oferty z train": label,
        "Precision@10": precision,
        "Recall@10": recall,
        "czas treningu [s]": fit_time,
        "czas predykcji [s]": predict_time,
    })

results = pd.DataFrame(results).round(4)
print(results.to_string(index=False))

oferty z train  Precision@10  Recall@10  czas treningu [s]  czas predykcji [s]
nie wykluczamy        0.0019     0.0032             0.0366              0.2138
    wykluczamy        0.0014     0.0025             0.0366              1.5478
